# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bilalahmed251/-ML-Search-Discovery/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [60]:
# This cell is for CODE (numbers, a query, a check).
# Finding 1: Pages with weaker search performance signals, such as lower CTR or weaker average position, may be useful candidates for review. The label in this project comes from the observed trend_direction field, where pages marked down are labeled as declining. This supports a directional ranking claim, but it does not prove that refreshing the page will cause performance to improve.

# Finding 2: A machine-learning ranking model can prioritize pages for content-refresh review. The validation design should use an honest grouped or time-aware split so that pages from the same client or future observations do not make the result look better than it really is. The measured Precision@50 is evidence about ranking quality on the tested sample, not a guarantee of business impact.

# Methodology questions: Are the labels measured consistently? Are the features available before the prediction moment? Does performance remain acceptable when pages from an unseen client are tested? Could seasonality, tracking quality, or changes in search demand affect the observed label?



In [61]:
# This cell records the methodology questions checked in Section 1.

methodology_questions = [
    "Where does the label come from?",
    "Are all features available before prediction?",
    "Does the validation split prevent client memorization?",
    "Could seasonality or tracking quality affect the label?",
    "Does Precision@50 measure ranking quality rather than causal impact?"
]

for number, question in enumerate(methodology_questions, start=1):
    print(f"{number}. {question}")


1. Where does the label come from?
2. Are all features available before prediction?
3. Does the validation split prevent client memorization?
4. Could seasonality or tracking quality affect the label?
5. Does Precision@50 measure ranking quality rather than causal impact?


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [62]:
# This cell is for CODE (numbers, a query, a check).
# The Week-5 random stratified split produced a strong measured Precision@50, but pages from the same client could appear in both training and testing. For a more honest audit, I will use a grouped split when a client identifier is available. Entire clients will be assigned to either training or testing, so the model must generalize to unseen clients. I will compare the before result from Week 5 with the after result from this grouped split. If no usable client field exists, I will document that limitation rather than inventing a grouping variable.


In [63]:
import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score

# Load the dataset
repo = "/content/ML-Search-Discovery"

if not os.path.exists(repo):
    !git clone -q --depth 1 https://github.com/bilalahmed251/-ML-Search-Discovery.git {repo}

df_audit = pd.read_csv(
    f"{repo}/data/raw/content_refresh_anonymized.csv"
 ).copy()

# Create the observed target
df_audit["is_declining_label"] = (
    df_audit["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Find a usable client/group column
possible_group_columns = [
    "client_id",
    "client",
    "account_id",
    "group_id"
]

group_column = next(
    (
        col for col in possible_group_columns
        if col in df_audit.columns
        and df_audit[col].dropna().nunique() > 1
    ),
    None
)

print("Group column found:", group_column)

# Rebuild the same Week-5 feature set
numeric_features_audit = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "scroll_rate",
]

categorical_features_audit = [
    "content_type",
    "main_intent",
    "position_tier",
    "impression_tier",
]

numeric_features_audit = [
    col for col in numeric_features_audit
    if col in df_audit.columns
]

categorical_features_audit = [
    col for col in categorical_features_audit
    if col in df_audit.columns
]

X_num_audit = df_audit[numeric_features_audit].apply(
    pd.to_numeric,
    errors="coerce"
).copy()

X_num_audit = X_num_audit.fillna(
    X_num_audit.median()
)

X_cat_audit = pd.get_dummies(
    df_audit[categorical_features_audit].fillna("MISSING"),
    columns=categorical_features_audit,
    dtype=int
)

X_audit = pd.concat(
    [X_num_audit, X_cat_audit],
    axis=1
)

y_audit = df_audit["is_declining_label"]

# Use grouped split if a valid group column exists
if group_column is not None:
    groups = df_audit[group_column].astype(str)

    group_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=42
    )

    train_positions, test_positions = next(
        group_splitter.split(X_audit, y_audit, groups=groups)
    )

    X_train_honest = X_audit.iloc[train_positions]
    X_test_honest = X_audit.iloc[test_positions]
    y_train_honest = y_audit.iloc[train_positions]
    y_test_honest = y_audit.iloc[test_positions]

    print("Split type: grouped by", group_column)
    print("Training groups:", groups.iloc[train_positions].nunique())
    print("Testing groups:", groups.iloc[test_positions].nunique())

else:
    raise ValueError(
        "No usable client/group column was found. "
        "Do not invent a grouping variable; document this limitation."
    )

# Train Random Forest on grouped training data
honest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

honest_model.fit(
    X_train_honest,
    y_train_honest
)

# Rank unseen test-group pages
honest_scores = honest_model.predict_proba(
    X_test_honest
)[:, 1]

def precision_at_k_audit(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_indices = np.argsort(scores)[::-1][:k]

    return float(
        y_true[top_indices].mean()
    )

honest_p50 = precision_at_k_audit(
    y_test_honest.to_numpy(),
    honest_scores,
    k=50
)

# Week-5 before result recorded from the completed notebook
week5_random_p50 = 0.94

before_after = pd.DataFrame({
    "evaluation": [
        "Week-5 random stratified split",
        "Week-6 grouped honest split"
    ],
    "precision_at_50": [
        week5_random_p50,
        honest_p50
    ]
})

display(before_after)

print("Before — Week-5 random split:",
      round(week5_random_p50, 3))

print("After — Week-6 grouped split:",
      round(honest_p50, 3))


Group column found: client_id
Split type: grouped by client_id
Training groups: 25
Testing groups: 7


,evaluation,precision_at_50
0,Week-5 random stratified split,0.94
1,Week-6 grouped honest split,0.70


Before — Week-5 random split: 0.94
After — Week-6 grouped split: 0.7


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [64]:
# This cell is for CODE (numbers, a query, a check).
# I audited the final feature set for target leakage, future-looking information, identifiers, and privacy-sensitive fields. The observed outcome fields trend_direction and trend_pct are excluded from X. The derived label is stored separately in y. The client_id is used only to create the grouped validation split and is not used as a model feature. No future outcome or action field is included.



In [68]:
# Fields that must never be model features
forbidden_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "future_trend",
    "refresh_flag",
    "action_taken",
    "outcome",
    "label",
    "url",
    "private_query",
]

# Original raw feature names used before one-hot encoding
raw_feature_names = (
    numeric_features_audit +
    categorical_features_audit
)

# Check forbidden raw fields
forbidden_in_raw_features = [
    field for field in forbidden_fields
    if field in raw_feature_names
]

leakage_keywords = [
    "trend",
    "future",
    "outcome",
    "label",
    "action",
    "refresh",
    "private",
    "url"
]

possible_leakage_in_final_X = [
    col for col in X_audit.columns
    if any(
        col.lower().startswith(prefix)
        for prefix in leakage_prefixes
    )
]

# Check whether client_id was used as a model feature
client_used_as_feature = (
    "client_id" in raw_feature_names or
    any(
        "client_id" in col.lower()
        for col in X_audit.columns
    )
)

print("Raw features checked:", len(raw_feature_names))
print("Final encoded features checked:", len(X_audit.columns))
print(
    "Forbidden fields in raw features:",
    forbidden_in_raw_features
)
print(
    "Possible leakage names in final X:",
    possible_leakage_in_final_X
)
print(
    "Client ID used as feature:",
    client_used_as_feature
)

leakage_passed = (
    len(forbidden_in_raw_features) == 0
    and len(possible_leakage_in_final_X) == 0
    and client_used_as_feature is False
)

print("Leakage audit passed:", leakage_passed)


Raw features checked: 14
Final encoded features checked: 27
Forbidden fields in raw features: []
Possible leakage names in final X: []
Client ID used as feature: False
Leakage audit passed: True


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [66]:
# This cell is for CODE (numbers, a query, a check).
# Original bold claim: The Random Forest is much better than the hand-written baseline and can identify the pages that should be refreshed.

# Safer claim: On this anonymized dataset and the tested evaluation split, the Random Forest achieved a higher measured Precision@50 than the hand-written baseline. Under the grouped client-level validation, the result should be interpreted as directional decision-support rather than proof that the model will improve traffic or that refreshing a selected page will cause better performance. Further time-based validation and real refresh experiments would be needed to measure business impact.



In [67]:
claim_audit = {
    "dataset": "Anonymized page-level search-performance dataset",
    "week5_random_split_precision_at_50": 0.94,
    "validation_design": "Grouped client-level split in Week 6",
    "claim_type": "Directional decision-support",
    "causal_claim_made": False,
    "business_impact_proven": False,
    "next_validation_needed": [
        "Time-based validation",
        "Prospective refresh experiment",
        "Post-refresh performance measurement"
    ]
}

for key, value in claim_audit.items():
    print(f"{key}:")
    print(value)
    print()


dataset:
Anonymized page-level search-performance dataset

week5_random_split_precision_at_50:
0.94

validation_design:
Grouped client-level split in Week 6

claim_type:
Directional decision-support

causal_claim_made:
False

business_impact_proven:
False

next_validation_needed:
['Time-based validation', 'Prospective refresh experiment', 'Post-refresh performance measurement']



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.